In [4]:
import os
import glob
import subprocess
import sys
import platform

def run_command(command, description=""):
    """Run a system command and return True if successful"""
    if description:
        print(f"🔧 {description}...")
    
    try:
        result = subprocess.run(command, shell=True, capture_output=True, text=True)
        if result.returncode == 0:
            if description:
                print(f"✅ {description} completed successfully")
            return True
        else:
            if description:
                print(f"❌ {description} failed: {result.stderr}")
            return False
    except Exception as e:
        if description:
            print(f"❌ {description} error: {e}")
        return False

def install_package(package_name):
    """Install a Python package if not already installed"""
    try:
        # Try to import the package first
        if package_name == "numpy":
            import numpy as np
        elif package_name == "matplotlib":
            import matplotlib.pyplot as plt
        elif package_name == "rasterio":
            import rasterio
        print(f"✅ {package_name} is already installed")
        return True
    except ImportError:
        print(f"⚠️ {package_name} not found. Installing...")
        try:
            # Install the package using pip
            subprocess.check_call([sys.executable, "-m", "pip", "install", package_name])
            print(f"✅ Successfully installed {package_name}")
            return True
        except subprocess.CalledProcessError:
            print(f"❌ Failed to install {package_name}")
            return False

def install_gdal_alternative():
    """Try to install rasterio as an alternative to GDAL"""
    print("🔧 Trying to install rasterio as an alternative to GDAL...")
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "rasterio"])
        import rasterio
        print("✅ Successfully installed rasterio")
        return True
    except (subprocess.CalledProcessError, ImportError):
        print("❌ Failed to install rasterio")
        return False

def main():
    print("🔍 Checking required packages...")
    
    # Install numpy first as it's a dependency for others
    if not install_package("numpy"):
        print("❌ Cannot continue without numpy")
        return
    
    # Install matplotlib
    if not install_package("matplotlib"):
        print("⚠️ Continuing without matplotlib - plots will not be generated")
        plot_available = False
    else:
        plot_available = True
    
    # Try to install GDAL or use rasterio as alternative
    gdal_available = False
    rasterio_available = False
    
    try:
        from osgeo import gdal
        print("✅ GDAL is already installed")
        gdal_available = True
        # Enable GDAL exceptions
        gdal.UseExceptions()
    except ImportError:
        print("⚠️ GDAL not found. Trying to install system dependencies first...")
        
        # Try to install system dependencies for GDAL
        system = platform.system()
        if system == "Linux":
            # Try to install GDAL system dependencies
            if run_command("apt-get update", "Updating package list"):
                if run_command("apt-get install -y libgdal-dev gdal-bin", "Installing GDAL system libraries"):
                    # Now try to install Python GDAL bindings
                    try:
                        subprocess.check_call([sys.executable, "-m", "pip", "install", "GDAL"])
                        from osgeo import gdal
                        gdal.UseExceptions()
                        gdal_available = True
                        print("✅ Successfully installed GDAL with system dependencies")
                    except (subprocess.CalledProcessError, ImportError):
                        print("❌ Failed to install Python GDAL bindings")
        
        # If GDAL installation failed, try rasterio as alternative
        if not gdal_available:
            print("⚠️ Falling back to rasterio as an alternative to GDAL...")
            rasterio_available = install_gdal_alternative()
    
    if not gdal_available and not rasterio_available:
        print("❌ Cannot continue without GDAL or rasterio")
        print("💡 Please try installing manually:")
        print("   For GDAL: sudo apt-get install libgdal-dev gdal-bin && pip install GDAL")
        print("   For rasterio: pip install rasterio")
        return
    
    # Now import the packages we need
    import numpy as np
    if plot_available:
        import matplotlib.pyplot as plt
    
    if rasterio_available:
        import rasterio
        from rasterio.plot import show

    # Create results directory
    results_dir = '../results'
    os.makedirs(results_dir, exist_ok=True)

    # Recursively get all .tif files
    tif_files = glob.glob('../data/**/*.tif', recursive=True)
    print(f"[FOUND {len(tif_files)} .tif FILES IN '../data']\n")

    if not tif_files:
        print("❌ No TIFF files found in ../data directory")
        print("💡 Please ensure your data is in the correct location")
        print("💡 Current working directory:", os.getcwd())
        
        # Check if data directory exists
        data_dir = '../data'
        if not os.path.exists(data_dir):
            print(f"💡 Data directory '{data_dir}' does not exist")
            print("💡 Creating empty data directory for testing")
            os.makedirs(data_dir, exist_ok=True)
            print("💡 Please add your TIFF files to the data directory and run again")
        return

    for file_path in tif_files:
        print(f"[PROCESSING FILE]: {file_path}")
        try:
            if gdal_available:
                ds = gdal.Open(file_path)
                if ds is None:
                    print(f"  ❌ Failed to open: {file_path}")
                    continue
                
                base_name = os.path.basename(file_path).replace('.tif', '')
                band_count = ds.RasterCount
                print(f"  [ RASTER BAND COUNT ]: {band_count}")

                bands = []
                for i in range(1, band_count + 1):
                    band = ds.GetRasterBand(i).ReadAsArray().astype(float)
                    bands.append(band)
                    
            elif rasterio_available:
                with rasterio.open(file_path) as src:
                    base_name = os.path.basename(file_path).replace('.tif', '')
                    band_count = src.count
                    print(f"  [ RASTER BAND COUNT ]: {band_count}")

                    bands = []
                    for i in range(1, band_count + 1):
                        band = src.read(i).astype(float)
                        bands.append(band)
                    
                    # Store metadata for later use
                    profile = src.profile
                    transform = src.transform
                    crs = src.crs

            # Compute and print stats
            for idx, band in enumerate(bands, start=1):
                # Check if band has valid data (not all NaN or zeros)
                if np.all(np.isnan(band)) or np.all(band == 0):
                    print(f"  ⚠️ BAND {idx} has no valid data")
                    continue
                    
                stats = {
                    'min': np.nanmin(band),
                    'max': np.nanmax(band),
                    'mean': np.nanmean(band),
                    'std': np.nanstd(band)
                }
                print(f"  [ BAND {idx} STATS ] Min={stats['min']:.3f}, Max={stats['max']:.3f}, "
                      f"Mean={stats['mean']:.3f}, StdDev={stats['std']:.3f}")

            # NDVI calculation: NDVI = (NIR - Red) / (NIR + Red)
            if band_count >= 4:
                try:
                    # Different satellite sensors have different band arrangements
                    # For Landsat 8: Red is Band 4, NIR is Band 5
                    # For Sentinel-2: Red is Band 4, NIR is Band 8
                    red = bands[3]   # Band 4 (index 3)
                    nir = bands[4]   # Band 5 (index 4) - adjust based on your sensor
                    
                    # Handle division by zero and NaN values
                    denominator = (nir + red)
                    denominator[denominator == 0] = np.nan  # Set zero denominators to NaN
                    
                    ndvi = (nir - red) / denominator
                    
                    # Apply valid range for NDVI (-1 to 1)
                    ndvi[ndvi < -1] = -1
                    ndvi[ndvi > 1] = 1
                    
                    if plot_available:
                        ndvi_plot_path = os.path.join(results_dir, f"{base_name}_ndvi.png")
                        plt.figure(figsize=(10, 8))
                        plt.imshow(ndvi, cmap='RdYlGn', vmin=-1, vmax=1)
                        plt.colorbar(label='NDVI Value')
                        plt.title(f'NDVI: {base_name}')
                        plt.savefig(ndvi_plot_path, dpi=150, bbox_inches='tight')
                        plt.close()
                        print(f"  ✅ NDVI graph saved to: {ndvi_plot_path}")
                    
                    # Save NDVI as GeoTIFF
                    ndvi_tiff_path = os.path.join(results_dir, f"{base_name}_ndvi.tif")
                    
                    if gdal_available:
                        driver = gdal.GetDriverByName('GTiff')
                        ndvi_ds = driver.Create(ndvi_tiff_path, ds.RasterXSize, ds.RasterYSize, 1, gdal.GDT_Float32)
                        ndvi_ds.SetGeoTransform(ds.GetGeoTransform())
                        ndvi_ds.SetProjection(ds.GetProjection())
                        ndvi_ds.GetRasterBand(1).WriteArray(ndvi)
                        ndvi_ds.GetRasterBand(1).SetNoDataValue(-9999)
                        ndvi_ds = None  # Close the file
                    elif rasterio_available:
                        # Update profile for output
                        profile.update(
                            dtype=rasterio.float32,
                            count=1,
                            nodata=-9999
                        )
                        with rasterio.open(ndvi_tiff_path, 'w', **profile) as dst:
                            dst.write(ndvi.astype(rasterio.float32), 1)
                    
                    print(f"  ✅ NDVI GeoTIFF saved to: {ndvi_tiff_path}")
                    
                except Exception as e:
                    print(f"  ❌ Error calculating NDVI: {e}")
            else:
                print(f"  ⚠️ Not enough bands for NDVI (requires at least 4, found {band_count})")

            # NDWI calculation: NDWI = (Green - NIR) / (Green + NIR)
            if band_count >= 4:
                try:
                    green = bands[2]  # Band 3 (index 2) - Green band
                    
                    # Handle division by zero and NaN values
                    denominator = (green + nir)
                    denominator[denominator == 0] = np.nan  # Set zero denominators to NaN
                    
                    ndwi = (green - nir) / denominator
                    
                    # Apply valid range for NDWI (-1 to 1)
                    ndwi[ndwi < -1] = -1
                    ndwi[ndwi > 1] = 1
                    
                    if plot_available:
                        ndwi_plot_path = os.path.join(results_dir, f"{base_name}_ndwi.png")
                        plt.figure(figsize=(10, 8))
                        plt.imshow(ndwi, cmap='Blues', vmin=-1, vmax=1)
                        plt.colorbar(label='NDWI Value')
                        plt.title(f'NDWI: {base_name}')
                        plt.savefig(ndwi_plot_path, dpi=150, bbox_inches='tight')
                        plt.close()
                        print(f"  ✅ NDWI graph saved to: {ndwi_plot_path}")
                    
                    # Save NDWI as GeoTIFF
                    ndwi_tiff_path = os.path.join(results_dir, f"{base_name}_ndwi.tif")
                    
                    if gdal_available:
                        driver = gdal.GetDriverByName('GTiff')
                        ndwi_ds = driver.Create(ndwi_tiff_path, ds.RasterXSize, ds.RasterYSize, 1, gdal.GDT_Float32)
                        ndwi_ds.SetGeoTransform(ds.GetGeoTransform())
                        ndwi_ds.SetProjection(ds.GetProjection())
                        ndwi_ds.GetRasterBand(1).WriteArray(ndwi)
                        ndwi_ds.GetRasterBand(1).SetNoDataValue(-9999)
                        ndwi_ds = None  # Close the file
                    elif rasterio_available:
                        with rasterio.open(ndwi_tiff_path, 'w', **profile) as dst:
                            dst.write(ndwi.astype(rasterio.float32), 1)
                    
                    print(f"  ✅ NDWI GeoTIFF saved to: {ndwi_tiff_path}")
                    
                except Exception as e:
                    print(f"  ❌ Error calculating NDWI: {e}")
            else:
                print(f"  ⚠️ Not enough bands for NDWI (requires at least 4, found {band_count})")

            print("")
            
        except Exception as e:
            print(f"  ❌ Error processing {file_path}: {e}")
            print("")

if __name__ == "__main__":
    main()

🔍 Checking required packages...
✅ numpy is already installed
✅ matplotlib is already installed
⚠️ GDAL not found. Trying to install system dependencies first...
🔧 Updating package list...
❌ Updating package list failed: E: List directory /var/lib/apt/lists/partial is missing. - Acquire (13: Permission denied)

⚠️ Falling back to rasterio as an alternative to GDAL...
🔧 Trying to install rasterio as an alternative to GDAL...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.2/22.2 MB 24.5 MB/s eta 0:00:0000:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.2/102.2 kB 40.5 MB/s eta 0:00:00
✅ Successfully installed rasterio
[FOUND 0 .tif FILES IN '../data']

❌ No TIFF files found in ../data directory
💡 Please ensure your data is in the correct location
💡 Current working directory: /home/jovyan/work
💡 Data directory '../data' does not exist
💡 Creating empty data directory for testing
💡 Please add your TIFF files to the data directory and run again
